# Complete single-source analysis of a real multiwavelength light curve

This maintained notebook runs one compatible multiwavelength CSV through the
complete public PGMUVI workflow: validation, sampling and variability
inspection, independent period evidence for every observational channel,
temporal consensus, wavelength-derived constraints, a real two-dimensional GP
fit, predictions, plots, residuals, phase diagnostics, fitted parameters,
noise provenance, warnings, and a JSON-safe structured report.

The bundled `examples/data/10131+3049.csv` file is only the default input.
Replace `SOURCE_CSV` in the first code cell to analyse another compatible
source; source identity, row counts, observational channels, physical
wavelengths, duplicate-wavelength groups, output counts, and report metadata
are derived from the selected file.


## Scope and scientific boundaries

The workflow uses linear flux, with time in input dimension 0 and physical
wavelength in dimension 1. The notebook discovers observational channels,
physical wavelengths, and duplicated-wavelength groups from `SOURCE_CSV`.
All retained observational channels contribute to independent period evidence
and remain available to temporal consensus. When multiple channels share one
physical wavelength, the notebook derives and reports the exact-GP training
selection from the discovered group. This is a computational identifiability
choice, not an instrument-channel calibration.

The required fit uses `2DWavelengthDependent` with an RBF wavelength kernel. Set `GP_NUM_COMPONENTS = 1` to fit one quasi-periodic temporal component. Set `GP_NUM_COMPONENTS > 1` to fit exactly that many spectral-mixture components through `consensus_multicomp`. `LS_NUM_COMPONENTS` independently controls how many Lomb–Scargle candidates are retained per observational channel. Other model families are not automatically selected from a single source.

### Google Colab bootstrap

The first code cell detects Google Colab automatically. In Colab it clones the
PR174 branch into `/content/pgmuvi-repo`, installs that checkout with the active
kernel's Python interpreter, clears stale `pgmuvi` imports, and verifies that
the package and selected source resolve correctly. In a normal repository
checkout it performs no installation.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys
import warnings

PGMUVI_COLAB_GIT_REF = (
    "pr174-add-complete-single-source-analysis-notebook"
)
PGMUVI_REPOSITORY_URL = "https://github.com/ICSM/pgmuvi.git"
PGMUVI_COLAB_CHECKOUT = Path("/content/pgmuvi-repo")


def _is_google_colab():
    return importlib.util.find_spec("google.colab") is not None


def _run_bootstrap_command(command):
    print("+", " ".join(str(part) for part in command))
    subprocess.run(command, check=True)


def _find_local_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "pgmuvi").is_dir()
            and (
                candidate / "examples/data/10131+3049.csv"
            ).is_file()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate a PGMUVI repository checkout. "
        "Run this notebook from inside the repository, or use Google Colab."
    )


def _prepare_repository_root():
    if not _is_google_colab():
        return _find_local_repository_root()

    if not (PGMUVI_COLAB_CHECKOUT / ".git").is_dir():
        _run_bootstrap_command(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                PGMUVI_COLAB_GIT_REF,
                PGMUVI_REPOSITORY_URL,
                str(PGMUVI_COLAB_CHECKOUT),
            ]
        )
    else:
        print(
            "Reusing existing Colab checkout:",
            PGMUVI_COLAB_CHECKOUT,
        )

    _run_bootstrap_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(PGMUVI_COLAB_CHECKOUT),
        ]
    )
    return PGMUVI_COLAB_CHECKOUT.resolve()


IS_GOOGLE_COLAB = _is_google_colab()
REPOSITORY_ROOT = _prepare_repository_root()

repository_string = str(REPOSITORY_ROOT)
sys.path = [
    entry for entry in sys.path if entry != repository_string
]
sys.path.insert(0, repository_string)

for module_name in list(sys.modules):
    if module_name == "pgmuvi" or module_name.startswith("pgmuvi."):
        del sys.modules[module_name]

importlib.invalidate_caches()

expected_module = (
    REPOSITORY_ROOT / "pgmuvi/single_source_analysis.py"
)
if not expected_module.is_file():
    raise FileNotFoundError(
        f"The PR174 analysis module is missing: {expected_module}"
    )

import pgmuvi

if pgmuvi.__file__ is None:
    raise ImportError("PGMUVI resolved as an invalid namespace package.")

imported_package = Path(pgmuvi.__file__).resolve()
expected_package_directory = (
    REPOSITORY_ROOT / "pgmuvi"
).resolve()
if expected_package_directory not in imported_package.parents:
    raise ImportError(
        "PGMUVI was imported from the wrong location: "
        f"{imported_package}. Expected a module inside "
        f"{expected_package_directory}."
    )

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import torch
from gpytorch.utils.warnings import GPInputWarning

from pgmuvi.single_source_analysis import (
    SINGLE_SOURCE_ANALYSIS_STAGE_ORDER,
    build_per_observational_channel_period_evidence,
    build_phase_folded_prediction_summary,
    build_single_source_analysis_report,
    preserve_single_source_analysis_state,
    resolve_single_source_period_component_configuration,
    single_source_json_safe,
    summarize_single_source_constraint_diagnostics,
    summarize_single_source_fit_explanations,
    summarize_single_source_fit_quality,
    summarize_single_source_noise_provenance,
    summarize_single_source_observational_channels,
    validate_single_source_stage_records,
    write_single_source_analysis_report,
)
from pgmuvi.wavelength_constraint_tutorial import (
    build_tutorial_fit_summary,
    build_wavelength_constraint_position_rows,
    load_wavelength_constraint_tutorial_lightcurve,
    summarize_observational_channel_residuals,
    training_point_prediction_summary,
)

SEED = 174
RUN_REQUIRED_FIT = True
WRITE_JSON_REPORT = False
MAX_SAMPLES_PER_OBSERVATIONAL_CHANNEL = 50

# Apply PGMUVI's maintained per-physical-wavelength sampling gates.
# With SAMPLING_KWARGS=None, the package defaults are min_points=15,
# max_gap_fraction=0.3, min_baseline_factor=3.0, min_snr=3.0,
# and min_fraction_good_snr=0.5.
CHECK_SAMPLING = True
SAMPLING_KWARGS = None
TRAINING_ITER = 200
MINITER = 100
LS_NUM_COMPONENTS = 3
GP_NUM_COMPONENTS = 2
PERIOD_COMPONENT_CONFIGURATION = (
    resolve_single_source_period_component_configuration(
        ls_num_components=LS_NUM_COMPONENTS,
        gp_num_components=GP_NUM_COMPONENTS,
    )
)


OBSERVATIONAL_CHANNEL_MARKERS = (
    "o",
    "s",
    "^",
    "v",
    "D",
    "P",
    "X",
)


def build_observational_channel_styles(observational_channels):
    # Return deterministic color-marker styles in first-seen order.
    ordered_channels = list(
        dict.fromkeys(str(channel) for channel in observational_channels)
    )
    colors = tuple(
        plt.rcParams["axes.prop_cycle"].by_key().get("color", ())
    ) or ("C0",)
    style_combinations = [
        {"color": color, "marker": marker}
        for marker in OBSERVATIONAL_CHANNEL_MARKERS
        for color in colors
    ]
    return {
        observational_channel: dict(
            style_combinations[index % len(style_combinations)]
        )
        for index, observational_channel in enumerate(ordered_channels)
    }


def display_records(records, columns=None, title=None):
    "Display an export-safe Markdown table and return its source text."
    markdown_lines = []
    if title:
        markdown_lines.append(f"#### {title}")
        markdown_lines.append("")
    if not records:
        markdown_lines.append("_No records._")
        markdown_text = "\n".join(markdown_lines)
        display(Markdown(markdown_text))
        return markdown_text

    columns = list(columns or records[0])

    def _format_table_value(value):
        if value is None:
            text = "—"
        elif isinstance(value, float):
            text = f"{value:.6g}"
        elif isinstance(value, (list, tuple, set)):
            text = ", ".join(str(item) for item in value) or "—"
        else:
            text = str(value)
        return text.replace("\n", " ").replace("|", "\\|")

    markdown_lines.append(
        "| " + " | ".join(str(column) for column in columns) + " |"
    )
    markdown_lines.append(
        "| " + " | ".join("---" for _ in columns) + " |"
    )
    for record in records:
        markdown_lines.append(
            "| "
            + " | ".join(
                _format_table_value(record.get(column))
                for column in columns
            )
            + " |"
        )

    markdown_text = "\n".join(markdown_lines)
    display(Markdown(markdown_text))
    return markdown_text


# User configuration: replace this path with any compatible CSV.
SOURCE_CSV = REPOSITORY_ROOT / "examples/data/10131+3049.csv"
SOURCE_CSV = Path(SOURCE_CSV).expanduser().resolve()
SOURCE_ID = SOURCE_CSV.stem
REPORT_PATH = REPOSITORY_ROOT / (
    f"{SOURCE_ID}_single_source_analysis_report.json"
)
try:
    SOURCE_PATH_FOR_REPORT = str(
        SOURCE_CSV.relative_to(REPOSITORY_ROOT)
    )
except ValueError:
    SOURCE_PATH_FOR_REPORT = str(SOURCE_CSV)

if not SOURCE_CSV.is_file():
    raise FileNotFoundError(
        f"The selected source is missing: {SOURCE_CSV}"
    )

stage_records = []

{
    "environment": (
        "google_colab" if IS_GOOGLE_COLAB else "local_checkout"
    ),
    "repository_root": str(REPOSITORY_ROOT),
    "pgmuvi_import": str(imported_package),
    "source_id": SOURCE_ID,
    "source": SOURCE_PATH_FOR_REPORT,
    "required_fit": RUN_REQUIRED_FIT,
    "maximum_rows_per_observational_channel": (
        MAX_SAMPLES_PER_OBSERVATIONAL_CHANNEL
    ),
    "check_sampling": CHECK_SAMPLING,
    "sampling_kwargs": SAMPLING_KWARGS,
    "training_iter": TRAINING_ITER,
    "miniter": MINITER,
    "LS components requested": LS_NUM_COMPONENTS,
    "GP components requested": GP_NUM_COMPONENTS,
    "resolved fit strategy": PERIOD_COMPONENT_CONFIGURATION[
        "fit_strategy"
    ],
    "resolved time kernel": PERIOD_COMPONENT_CONFIGURATION[
        "time_kernel_type"
    ],
    "default_dtype_inside_analysis_stages": "torch.float64",
}


## 1. Load, validate, and inspect the retained source

Rows must have finite time, flux, uncertainty, and wavelength; flux and
uncertainty must both be strictly positive; and every row must retain a
non-empty observational-channel label. Deterministic time-stratified sampling
limits exact-GP cost without dropping any observational channel or physical
wavelength.

This notebook explicitly sets `check_sampling=True` and leaves `sampling_kwargs=None`, so PGMUVI uses its maintained defaults: `min_points=15`, `max_gap_fraction=0.3`, `min_baseline_factor=3.0`, `min_snr=3.0`, and `min_fraction_good_snr=0.5`. For this two-dimensional light curve, the checks are applied independently by physical wavelength. Failing wavelengths are removed with user-facing warnings, and loading stops only if no physical wavelength passes.


In [ ]:
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
    working_directory=REPOSITORY_ROOT,
):
    lightcurve, sampling_summary = (
        load_wavelength_constraint_tutorial_lightcurve(
            SOURCE_CSV,
            max_samples_per_observational_channel=(
                MAX_SAMPLES_PER_OBSERVATIONAL_CHANNEL
            ),
            check_sampling=CHECK_SAMPLING,
            sampling_kwargs=SAMPLING_KWARGS,
            name=f"{SOURCE_ID} complete single-source analysis",
        )
    )
    input_summary = summarize_single_source_observational_channels(
        lightcurve
    )

retained_time = (
    lightcurve.xdata[:, 0].detach().cpu().numpy().reshape(-1)
)
retained_wavelength = (
    lightcurve.xdata[:, 1].detach().cpu().numpy().reshape(-1)
)
retained_flux = (
    lightcurve.ydata.detach().cpu().numpy().reshape(-1)
)
assert lightcurve.yerr is not None
retained_flux_error = (
    lightcurve.yerr.detach().cpu().numpy().reshape(-1)
)
retained_observational_channel_array = np.asarray(
    lightcurve.observational_channel_labels,
    dtype=str,
).reshape(-1)
retained_observational_channels = list(
    dict.fromkeys(retained_observational_channel_array.tolist())
)
observational_channel_styles = (
    build_observational_channel_styles(
        retained_observational_channels
    )
)
assert set(observational_channel_styles) == set(
    retained_observational_channels
)
retained_physical_wavelengths = np.unique(retained_wavelength)
duplicate_wavelength_groups = input_summary[
    "duplicate_physical_wavelength_groups"
]



def duplicate_group_wavelength(group):
    for key in ("physical_wavelength", "wavelength"):
        if key in group:
            return float(group[key])
    raise KeyError(
        "Duplicate-wavelength summary lacks a wavelength coordinate."
    )
duplicate_wavelength_rows = [
    {
        "physical wavelength": duplicate_group_wavelength(group),
        "observational channels": ", ".join(
            group["observational_channels"]
        ),
        "default exact-GP selection": (
            group["observational_channels"][0]
        ),
        "ignored observational channels": ", ".join(
            group["observational_channels"][1:]
        ),
    }
    for group in duplicate_wavelength_groups
]

assert sampling_summary["check_sampling"] is CHECK_SAMPLING
assert sampling_summary["sampling_kwargs"] == {}
assert (
    sampling_summary["n_rows_before_sampling_quality_filter"]
    >= sampling_summary["n_rows_retained"]
)
assert (
    sampling_summary["n_rows_removed_by_sampling_quality_filter"]
    == sampling_summary[
        "n_rows_before_sampling_quality_filter"
    ]
    - sampling_summary["n_rows_retained"]
)
assert sampling_summary["n_rows_original"] > 0
assert (
    0
    <= sampling_summary["n_rows_excluded_by_validity_policy"]
    <= sampling_summary["n_rows_original"]
)
assert (
    sampling_summary["n_rows_eligible"]
    == sampling_summary["n_rows_original"]
    - sampling_summary["n_rows_excluded_by_validity_policy"]
)
assert (
    0
    < sampling_summary["n_rows_retained"]
    <= sampling_summary["n_rows_eligible"]
)

retained_row_count = retained_time.size
assert sampling_summary["n_rows_retained"] == retained_row_count
assert retained_wavelength.size == retained_row_count
assert retained_flux.size == retained_row_count
assert retained_flux_error.size == retained_row_count
assert (
    retained_observational_channel_array.size
    == retained_row_count
)
assert np.all(np.isfinite(retained_time))
assert np.all(np.isfinite(retained_wavelength))
assert np.all(np.isfinite(retained_flux))
assert np.all(np.isfinite(retained_flux_error))
assert np.all(retained_flux > 0.0)
assert np.all(retained_flux_error > 0.0)
assert np.all(
    np.char.str_len(retained_observational_channel_array) > 0
)

# Required input dimension order: ["time", "physical_wavelength"]
assert input_summary["dimension_order"] == [
    "time",
    "physical_wavelength",
]
assert input_summary["flux_domain"] == "linear"
assert input_summary["n_observational_channels"] == len(
    retained_observational_channels
)
assert input_summary["n_physical_wavelengths"] == len(
    retained_physical_wavelengths
)

observed_channels_by_wavelength = {}
for observational_channel, wavelength in zip(
    retained_observational_channel_array,
    retained_wavelength,
    strict=True,
):
    wavelength = float(wavelength)
    observed_channels_by_wavelength.setdefault(wavelength, [])
    if (
        observational_channel
        not in observed_channels_by_wavelength[wavelength]
    ):
        observed_channels_by_wavelength[wavelength].append(
            observational_channel
        )

derived_duplicate_groups = {
    wavelength: observational_channels
    for wavelength, observational_channels
    in observed_channels_by_wavelength.items()
    if len(observational_channels) > 1
}
assert len(duplicate_wavelength_groups) == len(
    derived_duplicate_groups
)
for group in duplicate_wavelength_groups:
    wavelength = duplicate_group_wavelength(group)
    reported_channels = list(group["observational_channels"])
    assert wavelength in derived_duplicate_groups
    assert reported_channels == derived_duplicate_groups[wavelength]
    assert len(reported_channels) > 1

stage_records.extend(
    [
        {"stage": "load_and_validate", "status": "completed"},
        {"stage": "sampling_and_variability", "status": "completed"},
    ]
)

if duplicate_wavelength_rows:
    display_records(
        duplicate_wavelength_rows,
        title=(
            "Automatically discovered duplicated physical wavelengths"
        ),
    )
else:
    display(
        Markdown(
            "_No duplicated physical wavelengths were discovered "
            "in this source._"
        )
    )

display_records(
    [
        {
            "original rows": sampling_summary["n_rows_original"],
            "eligible rows": sampling_summary["n_rows_eligible"],
            "retained rows": sampling_summary["n_rows_retained"],
            "sampling check enabled": sampling_summary[
                "check_sampling"
            ],
            "rows removed by sampling quality": sampling_summary[
                "n_rows_removed_by_sampling_quality_filter"
            ],
            "observational channels": input_summary[
                "n_observational_channels"
            ],
            "physical wavelengths": input_summary[
                "n_physical_wavelengths"
            ],
            "flux domain": input_summary["flux_domain"],
        }
    ],
    title="Validated source and deterministic GP sample",
)


In [ ]:
input_time = lightcurve.xdata[:, 0].detach().cpu().numpy()
input_flux = lightcurve.ydata.detach().cpu().numpy()
input_channels = np.asarray(
    lightcurve.observational_channel_labels,
    dtype=str,
)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(12, 6))
    for observational_channel in dict.fromkeys(
        input_channels.tolist()
    ):
        channel_mask = input_channels == observational_channel
        style = observational_channel_styles[observational_channel]
        ax.scatter(
            input_time[channel_mask],
            input_flux[channel_mask],
            s=18,
            alpha=0.75,
            label=observational_channel,
            color=style["color"],
            marker=style["marker"],
        )
    ax.set_xlabel("Time")
    ax.set_ylabel("Linear flux")
    ax.set_yscale("log")
    ax.set_title(
        "Sampling-quality-retained observations by observational channel"
    )
    ax.legend(
        fontsize=7,
        ncol=2,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    fig.tight_layout()
    plt.show()


## 2. Independent period evidence for every observational channel

Lomb–Scargle and data-ACF diagnostics are calculated separately for every
retained observational channel discovered in the source. A failure in one
channel is recorded rather than silently removing evidence from the rest. The ACF is retained only as a comparison curve and does not identify peaks or components. LS and fitted-GP periods are overlaid later as references.

The user controls the retained LS-candidate count independently from the fitted GP component count. The shared GP period summary is drawn once, on a period axis; spectral-mixture fits show the summed PSD and every individual fitted component curve.


In [ ]:
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
):
    period_evidence = build_per_observational_channel_period_evidence(
        lightcurve,
        num_peaks=LS_NUM_COMPONENTS,
        single_threshold=0.05,
        nyquist_factor=5,
        acf_n_lags=50,
        minimum_points=5,
    )

assert period_evidence[
    "all_channels_retained_for_consensus"
] is True
assert period_evidence["n_observational_channels"] == len(
    retained_observational_channels
)
assert len(period_evidence["rows"]) == len(
    retained_observational_channels
)
assert (
    period_evidence["n_available"]
    + period_evidence["n_unavailable"]
    == period_evidence["n_observational_channels"]
)
assert {
    row["observational_channel"] for row in period_evidence["rows"]
} == set(retained_observational_channels)
assert {
    row["status"] for row in period_evidence["rows"]
}.issubset({"available", "unavailable"})
for row in period_evidence["rows"]:
    assert row["n_points"] > 0
    assert row["physical_wavelengths"]
    assert np.all(
        np.isfinite(
            np.asarray(
                row["physical_wavelengths"],
                dtype=float,
            )
        )
    )

stage_records.append(
    {
        "stage": "per_observational_channel_period_evidence",
        "status": "completed",
        "available": period_evidence["n_available"],
        "unavailable": period_evidence["n_unavailable"],
    }
)

period_rows = []
for row in period_evidence["rows"]:
    lomb_scargle = row.get("lomb_scargle") or {}
    periods = lomb_scargle.get("peak_periods") or []
    significant = lomb_scargle.get("significant") or []
    period_rows.append(
        {
            "observational channel": row["observational_channel"],
            "physical wavelength": row["physical_wavelengths"][0],
            "points": row["n_points"],
            "status": row["status"],
            "LS candidate periods": periods,
            "candidate significance": significant,
            "ACF curve available": row.get("acf") is not None,
            "reason": row.get("reason"),
        }
    )

display_records(period_rows, title="Per-observational-channel period evidence")


The multi-method comparison is generated only after the required GP fit has completed. At that stage, `Lightcurve.plot_period_diagnostic_comparison()` reuses the registered full Lomb–Scargle and data-ACF results and the fitted GP period summary. This avoids a second bespoke plotting path and preserves every resolved GP PSD feature and fitted kernel component. The Lomb–Scargle panel is rendered by `Lightcurve.plot_lomb_scargle_periodogram()`, using period on a logarithmic x-axis and Lomb–Scargle power on a linear y-axis. All retained LS candidates remain visible for multi-component fits.


## 3. Explicit GP fits and parameter-workflow comparison

This section performs **two real fits**. The calls to `Lightcurve.fit()` are
written out in full so that the model, consensus strategy, source-type
constraint set, optimizer controls, and parameter-workflow switch are visible
at the point of execution.

The comparison is deliberately named **parameter workflow enabled versus
disabled**. It is not described as a fully constrained versus unconstrained
comparison:

- both fits use the same temporal-consensus pipeline;
- both use `constraint_set="LPV"`;
- both retain the same consensus-derived temporal initialization and temporal
  constraints;
- both use the same observations, duplicate-channel policy, random seed,
  component count, optimizer, learning rate, and iteration budget;
- only `use_parameter_workflow` changes.

Before optimizer training, PGMUVI builds the data-derived parameter context and, when `use_parameter_workflow=True`, applies supported initial values and live constraints.

When enabled, PGMUVI builds and applies the model's data-derived parameter workflow before optimizer training, including supported initial values and live constraints.

With `use_parameter_workflow=True`, PGMUVI builds the schema-driven
data-derived parameter context and applies supported initial values and live
constraints, including wavelength-derived entries. With
`use_parameter_workflow=False`, that schema-driven application step is
skipped; model defaults, the LPV constraint set, explicit user inputs, and the
period-consensus handoff remain active.

The evidence preview below is computed before either fit and does not mutate
either fitted model. After fitting, the registered workflow report and
parameter-position table show what was actually applied.


In [ ]:
# Duplicate physical wavelengths use Lightcurve.fit()'s safe default:
# duplicate_wavelength_policy="first".
# The first observational channel encountered at each duplicated physical
# wavelength enters exact-GP training, and the fit reports selected and
# ignored channels.
#
# After inspecting duplicate_wavelength_groups, an explicit override can be
# added to both fit calls:
# duplicate_wavelength_policy="select",
# duplicate_wavelength_selection={
#     duplicated_wavelength: preferred_observational_channel,
# },

assert RUN_REQUIRED_FIT, "The default Run All path must execute the GP fit."
assert TRAINING_ITER > 0
assert MINITER > 0
assert MINITER <= TRAINING_ITER

RESOLVED_FIT_STRATEGY = PERIOD_COMPONENT_CONFIGURATION["fit_strategy"]
RESOLVED_TIME_KERNEL_TYPE = PERIOD_COMPONENT_CONFIGURATION[
    "time_kernel_type"
]
RESOLVED_NUM_MIXTURES = PERIOD_COMPONENT_CONFIGURATION[
    "fit_kwargs"
].get("num_mixtures")

# Build a read-only preview of the same data-derived context used by the
# parameter workflow. This happens before model training and applies nothing.
parameter_estimation_context = lightcurve.get_parameter_estimation_context()
wavelength_preview = parameter_estimation_context.wavelength_diagnostics
wavelength_mean_preview = (
    parameter_estimation_context.wavelength_mean_diagnostics
)
preview_bounds = (
    wavelength_preview.model_recommended_lengthscale_bounds
    or wavelength_preview.recommended_lengthscale_bounds
)
preview_initial = (
    wavelength_preview.model_recommended_lengthscale_initial
    or wavelength_preview.recommended_lengthscale_initial
)
display_records(
    [
        {
            "evidence target": "wavelength covariance lengthscale",
            "available": wavelength_preview.available,
            "distinct physical wavelengths": (
                wavelength_preview.n_distinct_wavelengths
            ),
            "estimated initial value": preview_initial,
            "estimated lower bound": (
                None if preview_bounds is None else preview_bounds[0]
            ),
            "estimated upper bound": (
                None if preview_bounds is None else preview_bounds[1]
            ),
            "coordinate space": (
                wavelength_preview.model_coordinate_space
            ),
            "recommendation method": (
                wavelength_preview.recommendation_method
            ),
        },
        {
            "evidence target": "wavelength-dependent mean",
            "available": wavelength_mean_preview.available,
            "distinct physical wavelengths": (
                wavelength_preview.n_distinct_wavelengths
            ),
            "estimated initial value": None,
            "estimated lower bound": None,
            "estimated upper bound": None,
            "coordinate space": (
                wavelength_mean_preview.metadata.get(
                    "quadratic_mean_wavelength_coordinate"
                )
            ),
            "recommendation method": (
                "2DWavelengthDependent quadratic-mean recommendation"
            ),
        },
    ],
    title="Pre-fit wavelength evidence preview (nothing applied yet)",
)

# The original retained-data object becomes the workflow-enabled fit.
workflow_enabled_lightcurve = lightcurve

# Load a second, independent Lightcurve through the identical maintained
# loader. This prevents model, likelihood, constraint, optimizer, or cache
# state from leaking between the two fits.
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
    working_directory=REPOSITORY_ROOT,
):
    workflow_disabled_lightcurve, workflow_disabled_sampling_summary = (
        load_wavelength_constraint_tutorial_lightcurve(
            SOURCE_CSV,
            max_samples_per_observational_channel=(
                MAX_SAMPLES_PER_OBSERVATIONAL_CHANNEL
            ),
            check_sampling=CHECK_SAMPLING,
            sampling_kwargs=SAMPLING_KWARGS,
            name=f"{SOURCE_ID}: parameter workflow disabled",
        )
    )

# Compare stable preprocessing contract fields rather than the entire
# diagnostic dictionary. Precision-sensitive derived diagnostics are
# validated through exact equality of the retained float64 tensors below.
controlled_sampling_summary_keys = (
    "check_sampling",
    "sampling_kwargs",
    "n_rows_original",
    "n_rows_excluded_by_validity_policy",
    "n_rows_eligible",
    "n_rows_before_sampling_quality_filter",
    "n_rows_removed_by_sampling_quality_filter",
    "n_rows_retained",
)
for summary_key in controlled_sampling_summary_keys:
    assert workflow_disabled_sampling_summary[summary_key] == (
        sampling_summary[summary_key]
    )
assert workflow_enabled_lightcurve.xdata.dtype == torch.float64
assert workflow_disabled_lightcurve.xdata.dtype == torch.float64
assert torch.equal(
    workflow_enabled_lightcurve.xdata,
    workflow_disabled_lightcurve.xdata,
)
assert torch.equal(
    workflow_enabled_lightcurve.ydata,
    workflow_disabled_lightcurve.ydata,
)
assert torch.equal(
    workflow_enabled_lightcurve.yerr,
    workflow_disabled_lightcurve.yerr,
)
assert np.array_equal(
    workflow_enabled_lightcurve.observational_channel_labels,
    workflow_disabled_lightcurve.observational_channel_labels,
)

# FIT A: schema-driven parameter initialization and constraints are enabled.
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
):
    with warnings.catch_warnings(record=True) as fit_warning_records:
        warnings.simplefilter("always")
        workflow_enabled_result = workflow_enabled_lightcurve.fit(
            model="2DWavelengthDependent",
            fit_strategy=RESOLVED_FIT_STRATEGY,
            time_kernel_type=RESOLVED_TIME_KERNEL_TYPE,
            wavelength_kernel_type="rbf",
            num_mixtures=RESOLVED_NUM_MIXTURES,
            constraint_set="LPV",
            training_iter=TRAINING_ITER,
            miniter=MINITER,
            optim="Adam",
            lr=0.03,
            learn_additional_noise=True,
            use_parameter_workflow=True,
            verbose=False,
        )

# FIT B: the schema-driven parameter workflow is skipped. The temporal
# consensus, LPV defaults, duplicate handling, and optimizer controls remain
# exactly the same as FIT A.
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
):
    with warnings.catch_warnings(
        record=True
    ) as workflow_disabled_warning_records:
        warnings.simplefilter("always")
        workflow_disabled_result = workflow_disabled_lightcurve.fit(
            model="2DWavelengthDependent",
            fit_strategy=RESOLVED_FIT_STRATEGY,
            time_kernel_type=RESOLVED_TIME_KERNEL_TYPE,
            wavelength_kernel_type="rbf",
            num_mixtures=RESOLVED_NUM_MIXTURES,
            constraint_set="LPV",
            training_iter=TRAINING_ITER,
            miniter=MINITER,
            optim="Adam",
            lr=0.03,
            learn_additional_noise=True,
            use_parameter_workflow=False,
            verbose=False,
        )

# Preserve the workflow-enabled fit as the primary analysis object used by all
# later sections and by the final single-source report.
lightcurve = workflow_enabled_lightcurve
fit_result = workflow_enabled_result
fit_warning_messages = [
    str(item.message) for item in fit_warning_records
]
workflow_disabled_warning_messages = [
    str(item.message)
    for item in workflow_disabled_warning_records
]

fit_history = lightcurve.get_fit_history()
latest_fit = fit_history[-1]
workflow_disabled_fit_history = (
    workflow_disabled_lightcurve.get_fit_history()
)
workflow_disabled_latest_fit = workflow_disabled_fit_history[-1]

consensus_diagnostics = single_source_json_safe(
    lightcurve.consensus_diagnostics
)
workflow_disabled_consensus_diagnostics = single_source_json_safe(
    workflow_disabled_lightcurve.consensus_diagnostics
)
period_summary = lightcurve.get_period_summary(
    n_peaks=GP_NUM_COMPONENTS,
    prefer_fitted_psd=True,
)
workflow_disabled_period_summary = (
    workflow_disabled_lightcurve.get_period_summary(
        n_peaks=GP_NUM_COMPONENTS,
        prefer_fitted_psd=True,
    )
)
consensus_period = float(period_summary["dominant_period"])

assert lightcurve.is_fitted
assert workflow_disabled_lightcurve.is_fitted
assert latest_fit["success"] is True
assert latest_fit["failed"] is False
assert workflow_disabled_latest_fit["success"] is True
assert workflow_disabled_latest_fit["failed"] is False
assert latest_fit["training_iter"] == TRAINING_ITER
assert workflow_disabled_latest_fit["training_iter"] == TRAINING_ITER
assert consensus_diagnostics["consensus_success"] is True
assert (
    workflow_disabled_consensus_diagnostics["consensus_success"]
    is True
)
assert lightcurve.get_parameter_workflow_report()["available"] is True
assert (
    workflow_disabled_lightcurve.get_parameter_workflow_report()[
        "available"
    ]
    is False
)
assert np.isfinite(consensus_period)
assert consensus_period > 0.0

stage_records.extend(
    [
        {
            "stage": "period_consensus",
            "status": "completed",
            "final_consensus_period": consensus_diagnostics.get(
                "final_consensus_period"
            ),
        },
        {
            "stage": "wavelength_evidence",
            "status": "completed",
        },
        {
            "stage": "apply_and_verify_constraints",
            "status": "completed",
        },
        {
            "stage": "gp_fit",
            "status": "completed",
            "training_iter": TRAINING_ITER,
            "elapsed_seconds": latest_fit["elapsed_seconds"],
        },
    ]
)

display_records(
    [
        {
            "fit": "parameter workflow enabled",
            "use_parameter_workflow": True,
            "fit_success": latest_fit["success"],
            "model_class": latest_fit["model_class"],
            "fit_strategy": latest_fit["fit_strategy"],
            "dominant period": period_summary["dominant_period"],
            "training_iter": latest_fit["training_iter"],
            "elapsed_seconds": latest_fit["elapsed_seconds"],
            "warning_count": len(fit_warning_messages),
        },
        {
            "fit": "parameter workflow disabled",
            "use_parameter_workflow": False,
            "fit_success": workflow_disabled_latest_fit["success"],
            "model_class": workflow_disabled_latest_fit[
                "model_class"
            ],
            "fit_strategy": workflow_disabled_latest_fit[
                "fit_strategy"
            ],
            "dominant period": workflow_disabled_period_summary[
                "dominant_period"
            ],
            "training_iter": workflow_disabled_latest_fit[
                "training_iter"
            ],
            "elapsed_seconds": workflow_disabled_latest_fit[
                "elapsed_seconds"
            ],
            "warning_count": len(
                workflow_disabled_warning_messages
            ),
        },
    ],
    title="Controlled fit execution",
)

lightcurve.register_period_diagnostic_evidence(
    period_evidence
)
period_diagnostic_comparisons = (
    lightcurve.plot_period_diagnostic_comparison(
        period_summary=period_summary,
        period_summary_kwargs={
            "x_axis": "period",
            "log_x": True,
            "show_components": True,
            "max_peaks_to_mark": GP_NUM_COMPONENTS,
        },
        show=True,
        strict=True,
    )
)
expected_period_diagnostic_channels = {
    row["observational_channel"]
    for row in period_evidence["rows"]
}
assert set(period_diagnostic_comparisons) == (
    expected_period_diagnostic_channels
)
assert {
    record["status"]
    for record in period_diagnostic_comparisons.values()
}.issubset({"plotted", "skipped"})
assert all(
    len(record["gp_psd_peaks"]) >= 1
    for record in period_diagnostic_comparisons.values()
    if record["status"] == "plotted"
)
assert all(
    record["period_summary_figure"] is not None
    for record in period_diagnostic_comparisons.values()
    if record["status"] == "plotted"
)


## 4. Verify and interpret the live parameter constraints

This section separates four statements that are easy to confuse:

1. **Constraint registered** means that the model received a live numerical
   interval.
2. **Initialization valid** means that the optimizer started inside that
   interval.
3. **Fitted value valid** means that training respected the live interval.
4. **Parameter identified** is a statistical and scientific claim that is
   **not established** by the first three checks.

For every workflow-controlled parameter, the first table reports the registered
interval, realized initialization, fitted value, fractional position within the
interval, and distance to the nearest bound. The interpretation table then
explains the parameter's model role, coordinate system, origin of its value and
interval, technical status, and boundary status.

The boundary labels use the fitted fractional position within the interval:

- **interior:** farther than 10% of the interval width from either bound;
- **near-bound:** within 10% of a bound, but not within 1%;
- **at-bound:** within 1% of a bound;
- **position unavailable:** the live constraint was respected but no finite
  fractional position could be computed.

A row is called **technically satisfactory** only when the constraint was
registered, initialization and the fitted value are inside it, and the fitted
value is interior. This wording does not mean that the parameter is well identified. Near-bound, at-bound, or otherwise failed rows require sensitivity
checks before scientific interpretation.

The noise table is separate because supplied measurement-error variance and
learned additional homoscedastic variance have different meanings; the learned
term supplements rather than replaces the reported measurement errors.


In [ ]:
constraint_rows = build_wavelength_constraint_position_rows(
    lightcurve,
    fit_result,
)
assert constraint_rows
assert any(row["applies_to"] == "covariance" for row in constraint_rows)
for row in constraint_rows:
    assert row["application_order"] == "constraint_then_value"
    assert row["constraint_registered"]
    assert row["value_initialized"]
    assert row["initial_inside_constraint"]
    assert row["fitted_inside_constraint"]

parameter_workflow_report = lightcurve.get_parameter_workflow_report()
parameter_constraint_diagnostics = (
    summarize_single_source_constraint_diagnostics(
        constraint_rows,
        parameter_workflow_report=parameter_workflow_report,
        near_bound_fraction=0.1,
        at_bound_fraction=0.01,
    )
)
constraint_interpretation_rows = parameter_constraint_diagnostics["rows"]
constraint_interpretation_summary = (
    parameter_constraint_diagnostics["summary"]
)
assert constraint_interpretation_rows
assert all(
    row["identification_status"]
    == "not_established_by_constraint_diagnostics"
    for row in constraint_interpretation_rows
)
noise_provenance = summarize_single_source_noise_provenance(lightcurve)
duplicate_wavelength_resolution = single_source_json_safe(
    lightcurve.duplicate_wavelength_channel_resolution
)

assert noise_provenance["available"] is True
assert noise_provenance["learn_additional_noise"] is True
assert noise_provenance[
    "additional_noise_is_not_measurement_error_replacement"
] is True
resolution_groups = duplicate_wavelength_resolution["groups"]
if duplicate_wavelength_groups:
    assert duplicate_wavelength_resolution["policy"] == "first"
    assert len(resolution_groups) == len(
        duplicate_wavelength_groups
    )
    expected_default_duplicate_selection = {
        duplicate_group_wavelength(group): (
            group["observational_channels"][0]
        )
        for group in duplicate_wavelength_groups
    }
    actual_default_duplicate_selection = {
        duplicate_group_wavelength(group): (
            group["selected_observational_channel"]
        )
        for group in resolution_groups
    }
    assert (
        actual_default_duplicate_selection
        == expected_default_duplicate_selection
    )
else:
    assert resolution_groups == []

display_records(
    constraint_rows,
    [
        "parameter",
        "applies_to",
        "lower_bound",
        "initial_value",
        "fitted_value",
        "upper_bound",
        "fractional_position_within_bounds",
        "minimum_distance_to_bound",
    ],
    "Registered and fitted parameter constraints",
)
display_records(
    constraint_interpretation_rows,
    [
        "parameter",
        "role",
        "coordinate_system",
        "technical_status",
        "boundary_status",
        "technically_satisfactory",
        "identification_status",
    ],
    "Technical constraint and boundary interpretation",
)
display_records(
    [
        {
            "overall technical status": (
                constraint_interpretation_summary["status"]
            ),
            "technically satisfactory rows": (
                constraint_interpretation_summary[
                    "n_technically_satisfactory"
                ]
            ),
            "rows requiring review": (
                constraint_interpretation_summary[
                    "n_requiring_review"
                ]
            ),
            "satisfactory parameters": (
                constraint_interpretation_summary[
                    "technically_satisfactory_parameters"
                ]
            ),
            "parameters requiring review": (
                constraint_interpretation_summary[
                    "parameters_requiring_review"
                ]
            ),
            "near-bound parameters": (
                constraint_interpretation_summary[
                    "near_bound_parameters"
                ]
            ),
            "at-bound parameters": (
                constraint_interpretation_summary[
                    "at_bound_parameters"
                ]
            ),
            "identification established": (
                constraint_interpretation_summary[
                    "identification_established"
                ]
            ),
        }
    ],
    title="Constraint interpretation summary",
)

parameter_interpretation_lines = [
    "#### Parameter-by-parameter interpretation",
    "",
]
for interpretation_row in constraint_interpretation_rows:
    parameter_interpretation_lines.append(
        "- **"
        + interpretation_row["parameter"]
        + "** ("
        + interpretation_row["role"]
        + "; "
        + interpretation_row["coordinate_system"]
        + "): "
        + interpretation_row["interpretation"]
        + " Value origin: "
        + str(interpretation_row["value_origin"])
        + ". Interval origin: "
        + str(interpretation_row["interval_origin"])
        + "."
    )
display(Markdown("\n".join(parameter_interpretation_lines)))

display_records(
    [
        {
            "likelihood": noise_provenance["likelihood_class"],
            "learn additional noise": noise_provenance[
                "learn_additional_noise"
            ],
            "fixed measurement variance": noise_provenance[
                "fixed_measurement_variance"
            ],
            "initial additional variance": noise_provenance[
                "initial_additional_noise_variance"
            ],
            "fitted additional variance": noise_provenance[
                "fitted_additional_noise_variance"
            ],
        }
    ],
    title="Noise provenance",
)


### 4.1 What this comparison does—and does not—measure

The following tables and figures compare the complete schema-driven parameter
workflow against the same fit with that workflow skipped. The disabled fit is **not unconstrained**: the LPV constraint set and consensus-derived temporal
constraints remain active in both fits. Consequently, differences below may be
attributed to the workflow's automatic parameter values and workflow-derived
constraints as a group, but not to wavelength constraints alone.

The numerical comparison includes the final and best objective, fitted dominant
period, fitted wavelength parameter, position inside its registered interval,
overall standardized-residual RMS, empirical 95% predictive coverage, and
per-channel fit-quality status. A signed difference is always defined as:

`parameter workflow enabled − parameter workflow disabled`.

The graphical comparison includes both optimization histories, per-channel
standardized-residual RMS, and every maintained fitted-light-curve panel for
both independent fits.


In [ ]:
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=GPInputWarning)
        workflow_enabled_predictions = (
            training_point_prediction_summary(
                workflow_enabled_lightcurve
            )
        )
        workflow_disabled_predictions = (
            training_point_prediction_summary(
                workflow_disabled_lightcurve
            )
        )

workflow_enabled_channel_rows = (
    summarize_observational_channel_residuals(
        workflow_enabled_predictions
    )
)
workflow_disabled_channel_rows = (
    summarize_observational_channel_residuals(
        workflow_disabled_predictions
    )
)
workflow_enabled_fit_quality = summarize_single_source_fit_quality(
    workflow_enabled_channel_rows,
    warning_threshold=2.0,
    severe_threshold=3.0,
)
workflow_disabled_fit_quality = summarize_single_source_fit_quality(
    workflow_disabled_channel_rows,
    warning_threshold=2.0,
    severe_threshold=3.0,
)


def _prediction_metrics(prediction_summary):
    standardized = np.asarray(
        prediction_summary["standardized_residual"],
        dtype=float,
    )
    finite = np.isfinite(standardized)
    if not np.any(finite):
        raise RuntimeError(
            "No finite standardized residuals are available."
        )
    standardized = standardized[finite]
    return {
        "standardized residual RMS": float(
            np.sqrt(np.mean(np.square(standardized)))
        ),
        "empirical 95% coverage": float(
            np.mean(np.abs(standardized) <= 1.959963984540054)
        ),
    }


def _constraint_arrays(lightcurve_object, parameter_name):
    parameters = lightcurve_object.get_parameters(
        raw=False,
        transform=False,
    )
    if parameter_name not in parameters:
        raise RuntimeError(
            f"Parameter {parameter_name!r} is unavailable."
        )
    values = np.asarray(
        parameters[parameter_name].detach().cpu(),
        dtype=float,
    ).reshape(-1)
    # get_parameters(raw=False) exposes model-relative, de-rawified
    # names such as ``mean_module.bias``. GPyTorch constraints are
    # registered on ``lightcurve_object.model`` against the corresponding
    # raw parameter name, for example ``mean_module.raw_bias``.
    constraint = None
    matched_model_parameter = False
    for raw_parameter_name, _ in (
        lightcurve_object.model.named_parameters()
    ):
        reported_parameter_name = ".".join(
            component.removeprefix("raw_")
            for component in raw_parameter_name.split(".")
        )
        if reported_parameter_name != parameter_name:
            continue
        matched_model_parameter = True
        candidate_constraint = (
            lightcurve_object.model.constraint_for_parameter_name(
                raw_parameter_name
            )
        )
        if candidate_constraint is not None:
            constraint = candidate_constraint
            break
    if not matched_model_parameter:
        raise RuntimeError(
            f"Could not resolve model parameter {parameter_name!r} "
            "to its registered raw parameter name."
        )
    if constraint is None:
        return values, None, None
    lower = np.asarray(
        constraint.lower_bound.detach().cpu(),
        dtype=float,
    ).reshape(-1)
    upper = np.asarray(
        constraint.upper_bound.detach().cpu(),
        dtype=float,
    ).reshape(-1)

    def _broadcast(array):
        if array.size == values.size:
            return array
        if array.size == 1:
            return np.repeat(array, values.size)
        raise RuntimeError(
            f"Constraint for {parameter_name!r} is not broadcastable."
        )

    return values, _broadcast(lower), _broadcast(upper)


def _quality_status_by_channel(fit_quality_summary):
    return {
        row["observational_channel"]: row["status"]
        for row in fit_quality_summary[
            "channels_requiring_review"
        ]
    }


workflow_enabled_metrics = _prediction_metrics(
    workflow_enabled_predictions
)
workflow_disabled_metrics = _prediction_metrics(
    workflow_disabled_predictions
)

workflow_parameters = list(
    dict.fromkeys(row["parameter"] for row in constraint_rows)
)
parameter_comparison_rows = []
for parameter_name in workflow_parameters:
    enabled_values, enabled_lower, enabled_upper = (
        _constraint_arrays(
            workflow_enabled_lightcurve,
            parameter_name,
        )
    )
    disabled_values, disabled_lower, disabled_upper = (
        _constraint_arrays(
            workflow_disabled_lightcurve,
            parameter_name,
        )
    )
    if enabled_values.size != disabled_values.size:
        raise RuntimeError(
            f"Parameter shape changed for {parameter_name!r}."
        )
    for component_index in range(enabled_values.size):
        enabled_fraction = (
            None
            if enabled_lower is None
            else (
                enabled_values[component_index]
                - enabled_lower[component_index]
            )
            / (
                enabled_upper[component_index]
                - enabled_lower[component_index]
            )
        )
        disabled_fraction = (
            None
            if disabled_lower is None
            else (
                disabled_values[component_index]
                - disabled_lower[component_index]
            )
            / (
                disabled_upper[component_index]
                - disabled_lower[component_index]
            )
        )
        parameter_comparison_rows.append(
            {
                "parameter": parameter_name,
                "component": component_index,
                "enabled fitted value": enabled_values[
                    component_index
                ],
                "disabled fitted value": disabled_values[
                    component_index
                ],
                "enabled − disabled": (
                    enabled_values[component_index]
                    - disabled_values[component_index]
                ),
                "enabled lower bound": (
                    None
                    if enabled_lower is None
                    else enabled_lower[component_index]
                ),
                "enabled upper bound": (
                    None
                    if enabled_upper is None
                    else enabled_upper[component_index]
                ),
                "enabled fractional position": enabled_fraction,
                "disabled lower bound": (
                    None
                    if disabled_lower is None
                    else disabled_lower[component_index]
                ),
                "disabled upper bound": (
                    None
                    if disabled_upper is None
                    else disabled_upper[component_index]
                ),
                "disabled fractional position": disabled_fraction,
            }
        )

covariance_parameter_names = list(
    dict.fromkeys(
        row["parameter"]
        for row in constraint_rows
        if row["applies_to"] == "covariance"
    )
)
primary_wavelength_parameter = covariance_parameter_names[0]
enabled_wavelength_values, _, _ = _constraint_arrays(
    workflow_enabled_lightcurve,
    primary_wavelength_parameter,
)
disabled_wavelength_values, _, _ = _constraint_arrays(
    workflow_disabled_lightcurve,
    primary_wavelength_parameter,
)
enabled_losses = np.asarray(
    workflow_enabled_result["loss"],
    dtype=float,
)
disabled_losses = np.asarray(
    workflow_disabled_result["loss"],
    dtype=float,
)

workflow_comparison_summary_rows = [
    {
        "fit": "parameter workflow enabled",
        "workflow report available": True,
        "final objective": enabled_losses[-1],
        "best objective": np.min(enabled_losses),
        "dominant period": period_summary["dominant_period"],
        "wavelength parameter": primary_wavelength_parameter,
        "fitted wavelength value": np.median(
            enabled_wavelength_values
        ),
        "standardized residual RMS": (
            workflow_enabled_metrics[
                "standardized residual RMS"
            ]
        ),
        "empirical 95% coverage": (
            workflow_enabled_metrics[
                "empirical 95% coverage"
            ]
        ),
        "fit-quality status": workflow_enabled_fit_quality[
            "status"
        ],
        "channels requiring review": (
            workflow_enabled_fit_quality[
                "n_channels_requiring_review"
            ]
        ),
    },
    {
        "fit": "parameter workflow disabled",
        "workflow report available": False,
        "final objective": disabled_losses[-1],
        "best objective": np.min(disabled_losses),
        "dominant period": workflow_disabled_period_summary[
            "dominant_period"
        ],
        "wavelength parameter": primary_wavelength_parameter,
        "fitted wavelength value": np.median(
            disabled_wavelength_values
        ),
        "standardized residual RMS": (
            workflow_disabled_metrics[
                "standardized residual RMS"
            ]
        ),
        "empirical 95% coverage": (
            workflow_disabled_metrics[
                "empirical 95% coverage"
            ]
        ),
        "fit-quality status": workflow_disabled_fit_quality[
            "status"
        ],
        "channels requiring review": (
            workflow_disabled_fit_quality[
                "n_channels_requiring_review"
            ]
        ),
    },
]
workflow_comparison_difference_rows = [
    {
        "difference definition": "enabled − disabled",
        "final objective": (
            enabled_losses[-1] - disabled_losses[-1]
        ),
        "best objective": (
            np.min(enabled_losses) - np.min(disabled_losses)
        ),
        "dominant period": (
            float(period_summary["dominant_period"])
            - float(
                workflow_disabled_period_summary[
                    "dominant_period"
                ]
            )
        ),
        "fitted wavelength value": (
            np.median(enabled_wavelength_values)
            - np.median(disabled_wavelength_values)
        ),
        "standardized residual RMS": (
            workflow_enabled_metrics[
                "standardized residual RMS"
            ]
            - workflow_disabled_metrics[
                "standardized residual RMS"
            ]
        ),
        "empirical 95% coverage": (
            workflow_enabled_metrics[
                "empirical 95% coverage"
            ]
            - workflow_disabled_metrics[
                "empirical 95% coverage"
            ]
        ),
        "channels requiring review": (
            workflow_enabled_fit_quality[
                "n_channels_requiring_review"
            ]
            - workflow_disabled_fit_quality[
                "n_channels_requiring_review"
            ]
        ),
    }
]

enabled_status = _quality_status_by_channel(
    workflow_enabled_fit_quality
)
disabled_status = _quality_status_by_channel(
    workflow_disabled_fit_quality
)
enabled_channel_map = {
    row["observational_channel"]: row
    for row in workflow_enabled_channel_rows
}
disabled_channel_map = {
    row["observational_channel"]: row
    for row in workflow_disabled_channel_rows
}
assert set(enabled_channel_map) == set(disabled_channel_map)

workflow_comparison_channel_rows = []
for observational_channel in enabled_channel_map:
    enabled_row = enabled_channel_map[observational_channel]
    disabled_row = disabled_channel_map[observational_channel]
    workflow_comparison_channel_rows.append(
        {
            "observational channel": observational_channel,
            "physical wavelength": enabled_row[
                "physical_wavelength"
            ],
            "enabled standardized RMS": enabled_row[
                "standardized_residual_rms"
            ],
            "disabled standardized RMS": disabled_row[
                "standardized_residual_rms"
            ],
            "enabled − disabled RMS": (
                enabled_row["standardized_residual_rms"]
                - disabled_row["standardized_residual_rms"]
            ),
            "enabled status": enabled_status.get(
                observational_channel,
                "pass",
            ),
            "disabled status": disabled_status.get(
                observational_channel,
                "pass",
            ),
        }
    )

display_records(
    workflow_comparison_summary_rows,
    title="Parameter-workflow fit comparison",
)
display_records(
    workflow_comparison_difference_rows,
    title="Signed numerical differences",
)
display_records(
    parameter_comparison_rows,
    title="Fitted workflow parameters and registered-bound positions",
)
display_records(
    workflow_comparison_channel_rows,
    title="Per-channel residual and fit-quality comparison",
)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(
        np.arange(1, enabled_losses.size + 1),
        enabled_losses,
        label="parameter workflow enabled",
    )
    ax.plot(
        np.arange(1, disabled_losses.size + 1),
        disabled_losses,
        label="parameter workflow disabled",
    )
    ax.set_yscale("log")
    ax.set_xlabel("Optimizer iteration")
    ax.set_ylabel("Negative marginal log likelihood")
    ax.set_title("Controlled optimization-history comparison")
    ax.legend()
    fig.tight_layout()
    plt.show()

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(10, 5))
    wavelengths = np.asarray(
        [
            row["physical wavelength"]
            for row in workflow_comparison_channel_rows
        ],
        dtype=float,
    )
    enabled_rms = np.asarray(
        [
            row["enabled standardized RMS"]
            for row in workflow_comparison_channel_rows
        ],
        dtype=float,
    )
    disabled_rms = np.asarray(
        [
            row["disabled standardized RMS"]
            for row in workflow_comparison_channel_rows
        ],
        dtype=float,
    )
    ax.scatter(
        wavelengths,
        enabled_rms,
        marker="o",
        label="parameter workflow enabled",
    )
    ax.scatter(
        wavelengths,
        disabled_rms,
        marker="x",
        label="parameter workflow disabled",
    )
    for row in workflow_comparison_channel_rows:
        ax.annotate(
            row["observational channel"],
            (
                row["physical wavelength"],
                row["enabled standardized RMS"],
            ),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
    ax.axhline(2.0, linestyle="--", linewidth=1)
    ax.axhline(3.0, linestyle=":", linewidth=1)
    ax.set_xlabel("Physical wavelength")
    ax.set_ylabel("Standardized residual RMS")
    ax.set_title("Per-channel fit-quality comparison")
    ax.legend()
    fig.tight_layout()
    plt.show()

workflow_comparison_plot_warnings = []
for comparison_label, comparison_lightcurve in (
    (
        "Parameter workflow enabled",
        workflow_enabled_lightcurve,
    ),
    (
        "Parameter workflow disabled",
        workflow_disabled_lightcurve,
    ),
):
    display(Markdown(f"#### {comparison_label}: all fitted-light-curve panels"))
    with preserve_single_source_analysis_state(seed=SEED):
        with warnings.catch_warnings(record=True) as comparison_plot_records:
            warnings.simplefilter("always")
            comparison_figures = comparison_lightcurve.plot(
                show=False,
                save=False,
                n_pred=140,
                annotate_provenance=True,
            )
    workflow_comparison_plot_warnings.extend(
        str(item.message) for item in comparison_plot_records
    )
    assert len(comparison_figures) == len(
        np.unique(
            workflow_enabled_predictions[
                "physical_wavelength"
            ]
        )
    )
    for comparison_figure in comparison_figures:
        display(comparison_figure)
        plt.close(comparison_figure)

parameter_workflow_comparison = single_source_json_safe(
    {
        "comparison_label": (
            "parameter workflow enabled versus disabled"
        ),
        "difference_definition": "enabled minus disabled",
        "controlled_properties": {
            "same_retained_observations": True,
            "same_duplicate_wavelength_policy": "first",
            "same_seed": SEED,
            "same_fit_strategy": RESOLVED_FIT_STRATEGY,
            "same_time_kernel_type": RESOLVED_TIME_KERNEL_TYPE,
            "same_num_mixtures": RESOLVED_NUM_MIXTURES,
            "same_constraint_set": "LPV",
            "same_optimizer": "Adam",
            "same_learning_rate": 0.03,
            "same_training_iter": TRAINING_ITER,
            "same_miniter": MINITER,
            "same_learn_additional_noise": True,
        },
        "important_limitation": (
            "The disabled fit still uses LPV defaults and "
            "consensus-derived temporal initialization and constraints; "
            "this is not a fully unconstrained fit."
        ),
        "summary_rows": workflow_comparison_summary_rows,
        "difference_rows": workflow_comparison_difference_rows,
        "parameter_rows": parameter_comparison_rows,
        "channel_rows": workflow_comparison_channel_rows,
        "enabled_parameter_workflow_report": (
            lightcurve.get_parameter_workflow_report()
        ),
        "disabled_parameter_workflow_report": (
            workflow_disabled_lightcurve.get_parameter_workflow_report()
        ),
        "plot_warnings": workflow_comparison_plot_warnings,
    }
)


In [ ]:
losses = np.asarray(fit_result["loss"], dtype=float)
assert losses.size >= MINITER
assert np.all(np.isfinite(losses))

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses)
    ax.set_xlabel("Optimizer iteration")
    ax.set_ylabel("Negative marginal log likelihood")
    ax.set_title("Required GP optimization history")
    fig.tight_layout()
    plt.show()


## 5. Predictions and maintained fitted-light-curve plots

Predictions are generated only after the fit has succeeded. The training-point
summary provides finite predictive means and variances and row-aligned residuals.
The maintained `Lightcurve.plot()` API is then called with provenance
annotations, and every fitted physical-wavelength panel is displayed. The
notebook does not use a bespoke four-panel subset.


In [ ]:
with preserve_single_source_analysis_state(
    seed=SEED,
    default_dtype=torch.float64,
):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=GPInputWarning)
        predictions = training_point_prediction_summary(lightcurve)

prediction_array_keys = (
    "time",
    "physical_wavelength",
    "observational_channel",
    "predictive_mean",
    "predictive_variance",
    "measurement_standard_deviation",
    "residual",
    "standardized_residual",
)
prediction_array_lengths = {
    key: len(np.asarray(predictions[key]).reshape(-1))
    for key in prediction_array_keys
}
assert len(set(prediction_array_lengths.values())) == 1
n_predictions = prediction_array_lengths["predictive_mean"]
assert n_predictions > 0
assert np.all(np.isfinite(predictions["time"]))
assert np.all(np.isfinite(predictions["physical_wavelength"]))
assert np.all(np.isfinite(predictions["predictive_mean"]))
assert np.all(np.isfinite(predictions["predictive_variance"]))
assert np.all(predictions["predictive_variance"] >= 0.0)
assert np.all(
    np.isfinite(
        predictions["measurement_standard_deviation"]
    )
)
assert np.all(
    predictions["measurement_standard_deviation"] >= 0.0
)
assert np.all(np.isfinite(predictions["residual"]))
assert np.all(np.isfinite(predictions["standardized_residual"]))

stage_records.append(
    {
        "stage": "predictions",
        "status": "completed",
        "n_predictions": len(predictions["predictive_mean"]),
    }
)


In [ ]:
with preserve_single_source_analysis_state(seed=SEED):
    with warnings.catch_warnings(record=True) as plot_warning_records:
        warnings.simplefilter("always")
        fitted_lightcurve_figures = lightcurve.plot(
            show=False,
            save=False,
            n_pred=140,
            annotate_provenance=True,
        )

plot_warning_messages = [
    str(item.message) for item in plot_warning_records
]
expected_gp_figure_count = len(
    np.unique(predictions["physical_wavelength"])
)
assert len(fitted_lightcurve_figures) == (
    expected_gp_figure_count
)
assert all(figure.axes for figure in fitted_lightcurve_figures)

for figure in fitted_lightcurve_figures:
    display(figure)
    plt.close(figure)

assert len(fitted_lightcurve_figures) == expected_gp_figure_count

stage_records.append(
    {
        "stage": "plots",
        "status": "completed",
        "figure_count": len(fitted_lightcurve_figures),
        "warning_count": len(plot_warning_messages),
    }
)


## 6. Scale-aware residual, phase, and channel-pair diagnostics

All fitted wavelengths passed the maintained sampling gates, but accepted
channels can still differ in cadence, baseline, phase coverage, gap structure,
and fitted cross-channel covariance support. Phase folding remains diagnostic;
it does not replace time in the GP. These are in-sample training-point
diagnostics, not held-out validation.

Raw residuals and raw RMSE retain the flux units of each observational channel.
They are therefore not intrinsically comparable across channels with different
flux scales. The primary combined diagnostics below use standardized residuals:
standardized-residual RMS versus wavelength, standardized residual versus time,
and standardized phase-folded residuals. Empirical 95% predictive coverage and
two additional dimensionless summaries—RMSE relative to absolute median flux
and RMSE relative to robust variability amplitude—are retained in the table and
structured report.

Raw RMSE remains visible only as a secondary plot with a logarithmic y-axis.
Signed raw residuals are not log-transformed; instead, every observational
channel receives its own small-multiple panel and independent y-range. This
prevents a high-flux channel from compressing the residual structure of all
other channels onto zero.

With the notebook default `GP_NUM_COMPONENTS = 2`, the active fit uses a
spectral-mixture temporal kernel rather than a single quasi-periodic coherence
envelope. If the control is changed to one component, the fit uses the
quasi-periodic family instead. In both cases, support is evaluated directly
from the fitted normalized covariance rather than inferred from one invented
coherence timescale.

The channel-pair diagnostics compare normalized phase shapes, flux levels,
amplitudes, temporal and phase overlap, and fitted wavelength-kernel
correlation. A large discrepancy between filters with comparable scalar
wavelengths is **not** labelled a calibration failure automatically. PGMUVI
currently uses one scalar wavelength per channel and does not integrate a cool
AGB spectrum through the full filter response. Similar phase shapes combined
with a large flux-level disagreement are therefore classified as a
**passband/SED or calibration incompatibility candidate**. The present model
cannot distinguish those explanations.


In [ ]:
channel_residual_rows = summarize_observational_channel_residuals(
    predictions
)
phase_diagnostics = build_phase_folded_prediction_summary(
    predictions,
    period=consensus_period,
)

training_observational_channels = list(
    dict.fromkeys(
        np.asarray(
            predictions["observational_channel"],
            dtype=str,
        ).tolist()
    )
)
expected_training_channel_count = len(
    training_observational_channels
)
assert expected_training_channel_count > 0
assert len(channel_residual_rows) == (
    expected_training_channel_count
)
assert len(
    phase_diagnostics["observational_channel_rows"]
) == expected_training_channel_count

residual_observational_channels = {
    row["observational_channel"]
    for row in channel_residual_rows
}
phase_observational_channels = {
    row["observational_channel"]
    for row in phase_diagnostics["observational_channel_rows"]
}
assert residual_observational_channels == set(
    training_observational_channels
)
assert phase_observational_channels == set(
    training_observational_channels
)
assert all(
    row["n_points"] > 0 for row in channel_residual_rows
)
assert sum(
    row["n_points"] for row in channel_residual_rows
) == n_predictions
assert all(
    np.isfinite(row["physical_wavelength"])
    and np.isfinite(row["rmse"])
    and np.isfinite(row["standardized_residual_rms"])
    for row in channel_residual_rows
)
assert all(
    "fractional_rmse_over_abs_median_flux" in row
    and "normalized_rmse_over_robust_amplitude" in row
    and "empirical_95_percent_coverage" in row
    for row in channel_residual_rows
)
assert "standardized_residual" in phase_diagnostics
assert all(
    "standardized_residual_rms" in row
    and "empirical_95_percent_coverage" in row
    for row in phase_diagnostics["observational_channel_rows"]
)

finite_standardized_residual = np.asarray(
    predictions["standardized_residual"],
    dtype=float,
)
finite_standardized_residual = finite_standardized_residual[
    np.isfinite(finite_standardized_residual)
]
assert finite_standardized_residual.size > 0
overall_standardized_residual_rms = float(
    np.sqrt(np.mean(np.square(finite_standardized_residual)))
)
overall_empirical_95_percent_coverage = float(
    np.mean(
        np.abs(finite_standardized_residual)
        <= 1.959963984540054
    )
)
residual_scale_policy = {
    "primary_cross_channel_metric": "standardized_residual_rms",
    "secondary_raw_metric": "rmse_with_logarithmic_y_axis",
    "combined_signed_metric": "standardized_residual",
    "phase_folded_signed_metric": "standardized_residual",
    "raw_signed_presentation": "per_channel_independent_y_ranges",
    "signed_log_transform_applied": False,
}

fit_quality = summarize_single_source_fit_quality(
    channel_residual_rows,
    warning_threshold=2.0,
    severe_threshold=3.0,
)
fit_quality_warning_messages = fit_quality["warning_messages"]
fit_explanations = summarize_single_source_fit_explanations(
    predictions,
    period=consensus_period,
    lightcurve=lightcurve,
)
fit_explanation_warning_messages = fit_explanations[
    "warning_messages"
]

display_records(
    fit_explanations["observational_channel_rows"],
    columns=[
        "observational_channel",
        "physical_wavelength",
        "n_points",
        "time_baseline",
        "median_positive_cadence",
        "maximum_gap_fraction",
        "phase_bin_coverage_fraction",
        "predictive_mean_bias",
        "rmse",
        "standardized_residual_rms",
        "median_predictive_standard_deviation",
        "empirical_95_percent_coverage",
        "nearest_observational_channel",
        "nearest_pair_interpretation",
        "fit_interpretation",
        "final_diagnostic_interpretation",
    ],
    title="Per-channel fit explanation diagnostics",
)

display_records(
    channel_residual_rows,
    columns=[
        "observational_channel",
        "physical_wavelength",
        "n_points",
        "rmse",
        "fractional_rmse_over_abs_median_flux",
        "normalized_rmse_over_robust_amplitude",
        "standardized_residual_rms",
        "empirical_95_percent_coverage",
    ],
    title="Scale-aware residual metrics by observational channel",
)

display_records(
    fit_explanations["pairs_requiring_interpretation"],
    columns=[
        "observational_channel_a",
        "observational_channel_b",
        "wavelength_separation",
        "absolute_median_flux_difference_dex",
        "robust_amplitude_ratio_a_over_b",
        "shared_phase_bins",
        "phase_overlap_fraction",
        "normalized_phase_shape_correlation",
        "temporal_overlap_fraction",
        "median_epoch_separation_in_periods",
        "fitted_wavelength_kernel_correlation",
        "phase_aligned_temporal_kernel_median_absolute_correlation",
        "interpretation",
    ],
    title="Channel pairs requiring scientific interpretation",
)

display_records(
    fit_quality["channels_requiring_review"],
    columns=[
        "observational_channel",
        "physical_wavelength",
        "n_points",
        "standardized_residual_rms",
        "status",
    ],
    title="Observational channels requiring fit-quality review",
)

stage_records.append(
    {
        "stage": "residuals",
        "status": "completed",
        "n_observational_channels": len(channel_residual_rows),
        "fit_quality_status": fit_quality["status"],
        "n_channels_requiring_review": fit_quality[
            "n_channels_requiring_review"
        ],
        "primary_cross_channel_metric": (
            residual_scale_policy["primary_cross_channel_metric"]
        ),
        "raw_signed_presentation": (
            residual_scale_policy["raw_signed_presentation"]
        ),
    }
)

prediction_channels = np.asarray(
    predictions["observational_channel"],
    dtype=str,
)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(12, 6))
    for observational_channel in dict.fromkeys(
        prediction_channels.tolist()
    ):
        channel_mask = prediction_channels == observational_channel
        style = observational_channel_styles[observational_channel]
        ax.scatter(
            predictions["time"][channel_mask],
            predictions["standardized_residual"][channel_mask],
            s=18,
            alpha=0.75,
            label=observational_channel,
            color=style["color"],
            marker=style["marker"],
        )
    ax.axhline(0.0, linewidth=1)
    ax.set_xlabel("Time")
    ax.set_ylabel("Standardized residual")
    ax.set_title(
        "Standardized residual versus time by GP-training channel"
    )
    ax.legend(
        fontsize=7,
        ncol=2,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(9, 5))
    for row in channel_residual_rows:
        observational_channel = row["observational_channel"]
        style = observational_channel_styles[observational_channel]
        ax.scatter(
            row["physical_wavelength"],
            row["standardized_residual_rms"],
            s=55,
            color=style["color"],
            marker=style["marker"],
        )
        ax.annotate(
            observational_channel,
            (
                row["physical_wavelength"],
                row["standardized_residual_rms"],
            ),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
    ax.axhline(1.0, linewidth=1, linestyle="--")
    ax.axhline(2.0, linewidth=1, linestyle=":")
    ax.axhline(3.0, linewidth=1, linestyle=":")
    ax.set_xlabel("Physical wavelength")
    ax.set_ylabel("Standardized residual RMS")
    ax.set_title(
        "Primary scale-independent residual summary by channel"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(9, 5))
    positive_rmse_rows = [
        row
        for row in channel_residual_rows
        if np.isfinite(row["rmse"]) and row["rmse"] > 0.0
    ]
    for row in positive_rmse_rows:
        observational_channel = row["observational_channel"]
        style = observational_channel_styles[observational_channel]
        ax.scatter(
            row["physical_wavelength"],
            row["rmse"],
            s=55,
            color=style["color"],
            marker=style["marker"],
        )
        ax.annotate(
            observational_channel,
            (row["physical_wavelength"], row["rmse"]),
            fontsize=7,
            xytext=(3, 3),
            textcoords="offset points",
        )
    if positive_rmse_rows:
        ax.set_yscale("log")
    else:
        ax.text(
            0.5,
            0.5,
            "No positive finite raw RMSE values are available.",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.set_xlabel("Physical wavelength")
    ax.set_ylabel("Training residual RMSE [channel flux units]")
    ax.set_title(
        "Secondary raw-RMSE summary (logarithmic y-axis)"
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(9, 5))
    phase_values = np.asarray(
        phase_diagnostics["phase"],
        dtype=float,
    )
    standardized_phase_residual_values = np.asarray(
        phase_diagnostics["standardized_residual"],
        dtype=float,
    )
    assert phase_values.size == prediction_channels.size
    assert (
        standardized_phase_residual_values.size
        == prediction_channels.size
    )
    for observational_channel in dict.fromkeys(
        prediction_channels.tolist()
    ):
        channel_mask = prediction_channels == observational_channel
        style = observational_channel_styles[observational_channel]
        ax.scatter(
            phase_values[channel_mask],
            standardized_phase_residual_values[channel_mask],
            s=18,
            alpha=0.75,
            label=observational_channel,
            color=style["color"],
            marker=style["marker"],
        )
    ax.axhline(0.0, linewidth=1)
    ax.set_xlabel("Consensus phase")
    ax.set_ylabel("Standardized residual")
    ax.set_title(
        "Standardized phase-folded residual by observational channel"
    )
    ax.legend(
        fontsize=7,
        ncol=2,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

with preserve_single_source_analysis_state(seed=SEED):
    n_residual_columns = min(3, expected_training_channel_count)
    n_residual_rows = int(
        np.ceil(expected_training_channel_count / n_residual_columns)
    )
    fig, residual_axes = plt.subplots(
        n_residual_rows,
        n_residual_columns,
        figsize=(5 * n_residual_columns, 3.2 * n_residual_rows),
        squeeze=False,
        sharex=False,
        sharey=False,
    )
    for channel_index, observational_channel in enumerate(
        training_observational_channels
    ):
        ax = residual_axes.flat[channel_index]
        channel_mask = prediction_channels == observational_channel
        style = observational_channel_styles[observational_channel]
        channel_wavelength = np.unique(
            np.asarray(
                predictions["physical_wavelength"],
                dtype=float,
            )[channel_mask]
        )
        ax.scatter(
            np.asarray(predictions["time"], dtype=float)[channel_mask],
            np.asarray(predictions["residual"], dtype=float)[channel_mask],
            s=18,
            alpha=0.75,
            color=style["color"],
            marker=style["marker"],
        )
        ax.axhline(0.0, linewidth=1)
        wavelength_text = (
            f"{channel_wavelength[0]:.6g}"
            if channel_wavelength.size == 1
            else "multiple"
        )
        ax.set_title(
            f"{observational_channel}\n"
            f"physical wavelength = {wavelength_text}"
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("Raw residual [channel flux units]")
    for unused_axis in residual_axes.flat[
        expected_training_channel_count:
    ]:
        unused_axis.set_visible(False)
    fig.suptitle(
        "Raw residuals in per-channel panels with independent y-ranges",
        y=1.01,
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)

with preserve_single_source_analysis_state(seed=SEED):
    fig, ax = plt.subplots(figsize=(11, 6))
    profile_rows = fit_explanations["normalized_phase_profile_rows"]
    for observational_channel in training_observational_channels:
        channel_profile_rows = [
            row for row in profile_rows
            if row["observational_channel"] == observational_channel
        ]
        if not channel_profile_rows:
            continue
        style = observational_channel_styles[observational_channel]
        ax.plot(
            [row["phase_bin_center"] for row in channel_profile_rows],
            [
                row["normalized_median_observed_flux"]
                for row in channel_profile_rows
            ],
            label=observational_channel,
            color=style["color"],
            marker=style["marker"],
        )
    ax.set_xlabel("Consensus phase")
    ax.set_ylabel("Normalized median observed flux")
    ax.set_title(
        "Normalized phase-shape comparison by GP-training channel"
    )
    ax.legend(
        fontsize=7,
        ncol=2,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## 7. Structured fit, provenance, warning, and failure outputs

The compact report is JSON-safe and records every scientific stage, the
per-channel period evidence, consensus diagnostics, wavelength constraints,
fit history, predictions, residuals, phase diagnostics, channel-level fit
quality, pairwise explanation diagnostics, noise provenance,
duplicate-wavelength resolution, warnings, and runtime metadata.

The interpretation fields are diagnostic rather than automatic rejection or
calibration decisions. In particular, the report records that a scalar
pivot-wavelength model cannot separate full passband/SED effects from
calibration or unit incompatibility. JSON export is available but disabled by
default so executing the public notebook does not write an untracked artifact
unexpectedly.


In [ ]:
fit_summary = build_tutorial_fit_summary(
    model_name="2DWavelengthDependent",
    lightcurve=lightcurve,
    fit_result=fit_result,
    sampling_summary=sampling_summary,
    constraint_rows=constraint_rows,
    predictions=predictions,
    warning_messages=fit_warning_messages,
)

stage_records.append(
    {
        "stage": "structured_outputs",
        "status": "completed",
    }
)
validated_stage_records = validate_single_source_stage_records(stage_records)
assert [
    record["stage"] for record in validated_stage_records
] == list(SINGLE_SOURCE_ANALYSIS_STAGE_ORDER)

analysis_report = build_single_source_analysis_report(
    source_id=SOURCE_ID,
    source_path=SOURCE_PATH_FOR_REPORT,
    stage_records=validated_stage_records,
    input_summary={
        "sampling": sampling_summary,
        "observational_channels": input_summary,
    },
    period_evidence=period_evidence,
    consensus={
        "period_summary": single_source_json_safe(period_summary),
        "consensus_diagnostics": consensus_diagnostics,
    },
    wavelength_evidence={
        "parameter_workflow_report": parameter_workflow_report,
    },
    constraint_diagnostics={
        "constraint_rows": constraint_rows,
        "interpretation": parameter_constraint_diagnostics,
    },
    fit_diagnostics={
        "fit_summary": fit_summary,
        "fit_history": fit_history,
        "loss": losses,
        "parameter_workflow_comparison": parameter_workflow_comparison,
    },
    prediction_diagnostics={
        "n_predictions": len(predictions["predictive_mean"]),
        "nonfinite_mean_count": int(
            np.count_nonzero(
                ~np.isfinite(predictions["predictive_mean"])
            )
        ),
        "nonfinite_variance_count": int(
            np.count_nonzero(
                ~np.isfinite(predictions["predictive_variance"])
            )
        ),
    },
    residual_diagnostics={
        "observational_channel_rows": channel_residual_rows,
        "overall_rmse": fit_summary["residual_rmse"],
        "overall_standardized_residual_rms": (
            overall_standardized_residual_rms
        ),
        "overall_empirical_95_percent_coverage": (
            overall_empirical_95_percent_coverage
        ),
        "scale_policy": residual_scale_policy,
        "fit_quality": fit_quality,
        "fit_explanations": fit_explanations,
    },
    phase_diagnostics=phase_diagnostics,
    noise_provenance=noise_provenance,
    duplicate_wavelength_resolution=duplicate_wavelength_resolution,
    warnings=(
        fit_warning_messages
        + plot_warning_messages
        + fit_quality_warning_messages
        + fit_explanation_warning_messages
    ),
    failures=[],
)

if WRITE_JSON_REPORT:
    written_report = write_single_source_analysis_report(
        REPORT_PATH,
        analysis_report,
    )
    print(f"Wrote {written_report}")

display_records(
    [
        {
            "schema": analysis_report["schema"],
            "source": analysis_report["source_id"],
            "completed stages": len(
                analysis_report["stage_records"]
            ),
            "consensus period": consensus_period,
            "fit iterations": fit_summary["n_iterations"],
            "GP training rows": fit_summary["gp_training_scope"][
                "n_observations"
            ],
            "consensus input rows": fit_summary["consensus_input_scope"][
                "n_observations"
            ],
            "objective improved": fit_summary["objective_improved"],
            "residual RMSE": fit_summary["residual_rmse"],
            "standardized residual RMS": (
                overall_standardized_residual_rms
            ),
            "empirical 95% coverage": (
                overall_empirical_95_percent_coverage
            ),
            "fit_quality_status": fit_quality["status"],
            "channels requiring review": fit_quality[
                "n_channels_requiring_review"
            ],
            "interpreted channel pairs": fit_explanations[
                "n_pairs_requiring_interpretation"
            ],
            "retained wavelength minimum": fit_explanations[
                "physical_wavelength_minimum"
            ],
            "retained wavelength maximum": fit_explanations[
                "physical_wavelength_maximum"
            ],
            "warning count": len(analysis_report["warnings"]),
            "failure count": len(analysis_report["failures"]),
        }
    ],
    title="Complete single-source analysis summary",
)


## Interpretation and limitations

This notebook demonstrates that the requested workflow can run on one selected
real source. It does not establish a population-level model preference, a
passband-integrated spectral model, or instrument-channel calibration.

A fitted wavelength parameter near a live constraint boundary is not evidence
that the source is achromatic.

Every fitted wavelength passed the maintained sampling gates, but retained
channels can still have unequal cadence, phase coverage, epoch overlap, and
fitted covariance support. Completion is therefore not equivalent to fit
adequacy. Cross-channel adequacy is judged primarily with standardized
residuals and predictive coverage; raw RMSE remains a channel-unit diagnostic
and is not interpreted as directly comparable across flux scales. Signed raw
residuals are shown only in per-channel panels with independent y-ranges. These
residual and coverage summaries are evaluated at the training points and are
not a substitute for held-out predictive validation. The default
spectral-mixture temporal kernel has no single coherence
timescale; the reported temporal-support values come from the fitted normalized
kernel covariance itself. The same diagnostic remains kernel-aware when the
one-component quasi-periodic control is selected.

For this executed source, the retained wavelength range and flux dynamic range
are derived from the data and recorded in the report. Filters with similar
scalar wavelengths may have very different widths and response profiles, and a
cool AGB spectrum can contain a steep continuum and deep molecular blanketing.
The current pivot-wavelength representation cannot model those bandpass
integrals. Consequently, similar normalized phase shapes plus a large flux
disagreement are reported as a passband/SED **or** calibration incompatibility
candidate, not as proof of a calibration failure.

For every automatically discovered duplicated physical wavelength, the
reported channel selection applies only to exact-GP training. All retained
channels in each duplicate group are analysed independently for period evidence
and remain available to consensus. Before accepting a fit scientifically,
inspect rejected channels, harmonic or alias ambiguity, optimizer history, live
constraints, additional-noise magnitude, per-channel residuals, predictive
coverage, normalized phase shapes, pairwise flux differences, and the stated
scalar-wavelength limitation.
